In [1]:
!pip install pennylane datasets


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 67.9 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
import pennylane as qml
from pennylane import numpy as np

# Charger le dataset
ds = load_dataset("Genius-Society/Pima")

# Extraire X et y
X = np.array([[s['Pregnancies'], s['Glucose'], s['BloodPressure'],
               s['SkinThickness'], s['Insulin'], s['BMI'],
               s['DiabetesPedigreeFunction'], s['Age']]
              for s in ds['train']], dtype=float)

y = np.array([s['Outcome'] for s in ds['train']])


/usr/local/lib/python3.12/dist-packages/pennylane/__init__.py:209: RuntimeWarning: PennyLane is not yet compatible with JAX versions > 0.6.2. You have version 0.7.2 installed. Please downgrade JAX to 0.6.2 to avoid runtime errors using python -m pip install jax~=0.6.0 jaxlib~=0.6.0
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/614 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/77 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/77 [00:00<?, ? examples/s]

In [3]:
def normalize(x):
    return x / np.max(x)



In [4]:
n_qubits = 8
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def entangler_enhanced_encoding(x):
    x = normalize(x)

    # 1️⃣ Superposition
    for i in range(n_qubits):
        qml.Hadamard(wires=i)

    # 2️⃣ Time evolution encoding
    for i in range(n_qubits):
        qml.RZ(x[i], wires=i)

    # 3️⃣ Entanglement (chaîne)
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])

    return qml.state()




In [5]:
state = entangler_enhanced_encoding(X[0])
print(state)



[0.00659104-0.06215149j 0.01592856-0.06043617j 0.01609779-0.06039132j
 0.00676511-0.06213279j 0.02221735-0.0584178j  0.03080909-0.05437877j
 0.03065664-0.05446485j 0.02205362-0.05847981j 0.05963904-0.01869318j
 0.06178229-0.00944449j 0.0618085 -0.00927139j 0.05969117-0.01852605j
 0.05308881-0.03298225j 0.05747187-0.02456082j 0.05740285-0.02472171j
 0.05299621-0.03313083j 0.05782771-0.02371089j 0.06075185-0.01467866j
 0.06079273-0.01450843j 0.0578939 -0.02354881j 0.06196164-0.00818566j
 0.06248661+0.00129358j 0.06248999+0.00111854j 0.06193847-0.0083592j
 0.0316074 -0.05391866j 0.03940936-0.04850931j 0.03954508-0.04839872j
 0.03175831-0.05382992j 0.01715476-0.06009962j 0.02605963-0.05680797j
 0.0259004 -0.05688075j 0.01698634-0.06014744j 0.04008628-0.04795143j
 0.04688662-0.04132669j 0.04700219-0.04119519j 0.04022045-0.04783896j
 0.05098595-0.03614807j 0.05587278-0.02800862j 0.0557941 -0.02816502j
 0.05088449-0.03629075j 0.0599453 +0.01768648j 0.05657489+0.02656184j
 0.05650027+0.0267202

In [6]:
# Transformation du dataset (X contient vos 8 features)
X_enhanced = np.array([np.abs(entangler_enhanced_encoding(row))**2 for row in X])
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Split des données
X_train_e, X_test_e, y_train, y_test = train_test_split(X_enhanced, y, test_size=0.2, random_state=42)

# Entraînement
clf_enhanced = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_enhanced.fit(X_train_e, y_train)

# Résultat
y_pred_e = clf_enhanced.predict(X_test_e)
print(f"Précision avec Entangler Enhanced Encoding : {accuracy_score(y_test, y_pred_e):.2%}")

Précision avec Entangler Enhanced Encoding : 65.04%
